In [ ]:
# Load all the csv files

import pandas as pd
from pathlib import Path

data_dir = Path('../source_data')

df_applicants = pd.read_csv(data_dir /'applicants.csv')
df_applicants_update = pd.read_csv(data_dir /'applicants_update.csv')
df_courses = pd.read_csv(data_dir / 'courses.csv')
df_preferences = pd.read_csv(data_dir / 'preferences.csv')
df_qualifications = pd.read_csv(data_dir / 'qualifications.csv')

df_dict = {"applicants": df_applicants,
           "applicants_update": df_applicants_update,
           "courses": df_courses,
           "preferences": df_preferences,
           "qualifications": df_qualifications}





In [ ]:
# Row count of each csv

for name, df in df_dict.items():
    print(f"{name}.csv: {len(df)} rows")


applicants.csv: 15 rows
applicants_update.csv: 6 rows
courses.csv: 14 rows
preferences.csv: 22 rows
qualifications.csv: 17 rows


In [ ]:
# Search for duplicates in primary keys
duplicate_checks = {
    "applicants.applicant_id": df_applicants["applicant_id"],
    "applicants_update.applicant_id": df_applicants_update["applicant_id"],
    "courses.course_code": df_courses["course_code"],
    "preferences.preference_id": df_preferences["preference_id"],
    "qualifications.qualification_id": df_qualifications["qualification_id"]
}

for name, col in duplicate_checks.items():
    print(f"{name} duplicates: {col.duplicated().sum()}")

duplicate_update_rows = df_applicants_update["applicant_id"].duplicated(keep=False)
print('\n', df_applicants_update[duplicate_update_rows])


applicants.applicant_id duplicates: 0
applicants_update.applicant_id duplicates: 1
courses.course_code duplicates: 0
preferences.preference_id duplicates: 0
qualifications.qualification_id duplicates: 0

    applicant_id first_name last_name date_of_birth                      email  \
0          1002      James  O'Connor    1999-07-22  james.oconnor@hotmail.com   
5          1002      James  O'Connor    1999-07-22  james.oconnor@hotmail.com   

       phone state  postcode created_date updated_date  
0  423456789   NSW      2010   2025-01-10   2025-02-15  
5  423456789   NSW      2010   2025-01-10   2025-02-15  


In [40]:
# Missing value checks
for name, df in df_dict.items():
    print(f"\nMissing values in {name}:")
    print(df.isna().sum())
    


Missing values in applicants:
applicant_id     0
first_name       0
last_name        0
date_of_birth    1
email            0
phone            1
state            0
postcode         0
created_date     0
updated_date     0
dtype: int64

Missing values in applicants_update:
applicant_id     0
first_name       0
last_name        0
date_of_birth    0
email            0
phone            0
state            0
postcode         0
created_date     0
updated_date     0
dtype: int64

Missing values in courses:
course_code         0
course_name         0
institution_code    0
institution_name    0
campus              0
study_mode          0
duration_years      0
atar_cutoff         1
csp_available       0
active_flag         0
dtype: int64

Missing values in preferences:
preference_id       0
applicant_id        0
course_code         0
preference_order    0
application_year    0
offer_status        2
offer_date          5
response            6
response_date       8
dtype: int64

Missing values in qu

In [41]:
# Unique value checks for important categorical fields

categorical_checks = {
    "applicants.state": df_applicants["state"],
    "applicants_update.state": df_applicants_update["state"],
    "courses.study_mode": df_courses["study_mode"],
    "courses.csp_available": df_courses["csp_available"],
    "courses.active_flag": df_courses["active_flag"],
    "qualifications.qualification_type": df_qualifications["qualification_type"],
    "qualifications.verified": df_qualifications["verified"],
    "preferences.offer_status": df_preferences["offer_status"],
    "preferences.response": df_preferences["response"],
}

for name, col in categorical_checks.items():
    print(f"\nUnique values in {name}:")
    print(col.dropna().unique())


Unique values in applicants.state:
['QLD' 'NSW' 'VIC']

Unique values in applicants_update.state:
['NSW' 'QLD']

Unique values in courses.study_mode:
['Full-time' 'full-time' 'Part-time']

Unique values in courses.csp_available:
['Y' 'N']

Unique values in courses.active_flag:
[1 0]

Unique values in qualifications.qualification_type:
['Year 12' 'Diploma' 'Bachelor' 'Certificate IV']

Unique values in qualifications.verified:
['Y' 'N']

Unique values in preferences.offer_status:
['Offered' 'Not Offered']

Unique values in preferences.response:
['Accepted' 'Declined' 'Pending']


In [42]:
# Orphan checks

# Preferences with applicant_id not found in either applicants file
all_applicant_ids = pd.concat([
    df_applicants["applicant_id"],
    df_applicants_update["applicant_id"]
]).drop_duplicates()

orphan_preferences_applicants = df_preferences[
    ~df_preferences["applicant_id"].isin(all_applicant_ids)
]

print("Preferences with missing applicant:")
print(orphan_preferences_applicants)


# Preferences with course_code not found in courses
orphan_preferences_courses = df_preferences[
    ~df_preferences["course_code"].isin(df_courses["course_code"])
]

print("\nPreferences with missing course:")
print(orphan_preferences_courses)


# Qualifications with applicant_id not found in either applicants file
orphan_qualifications_applicants = df_qualifications[
    ~df_qualifications["applicant_id"].isin(all_applicant_ids)
]

print("\nQualifications with missing applicant:")
print(orphan_qualifications_applicants)

Preferences with missing applicant:
Empty DataFrame
Columns: [preference_id, applicant_id, course_code, preference_order, application_year, offer_status, offer_date, response, response_date]
Index: []

Preferences with missing course:
Empty DataFrame
Columns: [preference_id, applicant_id, course_code, preference_order, application_year, offer_status, offer_date, response, response_date]
Index: []

Qualifications with missing applicant:
   qualification_id  applicant_id qualification_type institution_name  \
16             Q017          9999            Year 12   Unknown School   

    year_completed  gpa  atar_score verified  
16            2018  NaN        70.0        Y  


In [43]:
# Identify applicant update changes

# Deduplicate update rows first
df_applicants_update_deduped = df_applicants_update.drop_duplicates()

# Join initial applicants to update extract
applicant_comparison = df_applicants.merge(
    df_applicants_update_deduped,
    on="applicant_id",
    how="outer",
    suffixes=("_initial", "_update"),
    indicator=True
)

applicant_comparison

,applicant_id,first_name_initial,last_name_initial,date_of_birth_initial,email_initial,phone_initial,state_initial,postcode_initial,created_date_initial,updated_date_initial,first_name_update,last_name_update,date_of_birth_update,email_update,phone_update,state_update,postcode_update,created_date_update,updated_date_update,_merge
0,1001,Sarah,Mitchell,1998-03-15,sarah.mitchell@gmail.com,412345678.0,QLD,4000.0,2025-01-10,2025-01-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,1002,James,O'Connor,1999-07-22,james.oconnor@hotmail.com,423456789.0,QLD,4101.0,2025-01-10,2025-01-10,James,O'Connor,1999-07-22,james.oconnor@hotmail.com,423456789.0,NSW,2010.0,2025-01-10,2025-02-15,both
2,1003,Emily,Zhang,NaN,emily.zhang@outlook.com,434567890.0,QLD,4220.0,2025-01-11,2025-01-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,1004,Mohammed,Al-Rashid,2000-11-03,m.alrashid@gmail.com,445678901.0,NSW,2000.0,2025-01-11,2025-01-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,1005,Jessica,Brown,1999-02-28,jess.brown@yahoo.com,456789012.0,QLD,4350.0,2025-01-12,2025-01-12,Jessica,Brown-Taylor,1999-02-28,jess.browntaylor@gmail.com,456789012.0,QLD,4350.0,2025-01-12,2025-02-20,both
5,1006,Liam,Nguyen,2001-05-14,liam.nguyen@gmail.com,467890123.0,QLD,4670.0,2025-01-12,2025-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
6,1007,Olivia,Smith,2000-08-30,olivia.smith@outlook.com,NaN,QLD,4000.0,2025-01-13,2025-01-13,Olivia,Smith,2000-08-30,olivia.smith@outlook.com,478901234.0,QLD,4000.0,2025-01-13,2025-02-22,both
7,1008,Ethan,Williams,1998-12-01,ethan.w@gmail.com,489012345.0,QLD,4121.0,2025-01-13,2025-01-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
8,1009,Chloe,Patel,1999-09-17,chloe.patel@gmail.com,490123456.0,VIC,3000.0,2025-01-14,2025-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
9,1010,Noah,Garcia,2001-01-25,noah.garcia@hotmail.com,401234567.0,QLD,4500.0,2025-01-14,2025-01-14,Noah,Garcia,2001-01-25,noah.garcia@hotmail.com,401234567.0,QLD,4500.0,2025-01-14,2025-01-14,both


In [44]:
# Check which existing applicants changed

tracked_columns = [
    "first_name",
    "last_name",
    "date_of_birth",
    "email",
    "phone",
    "state",
    "postcode",
]

existing_applicants = applicant_comparison[
    applicant_comparison["_merge"] == "both"
].copy()

for col in tracked_columns:
    existing_applicants[f"{col}_changed"] = (
        existing_applicants[f"{col}_initial"].astype(str)
        != existing_applicants[f"{col}_update"].astype(str)
    )

change_columns = [f"{col}_changed" for col in tracked_columns]

existing_applicants[
    ["applicant_id"] 
    + [f"{col}_initial" for col in tracked_columns]
    + [f"{col}_update" for col in tracked_columns]
    + change_columns
]

,applicant_id,first_name_initial,last_name_initial,date_of_birth_initial,email_initial,phone_initial,state_initial,postcode_initial,first_name_update,last_name_update,...,phone_update,state_update,postcode_update,first_name_changed,last_name_changed,date_of_birth_changed,email_changed,phone_changed,state_changed,postcode_changed
1,1002,James,O'Connor,1999-07-22,james.oconnor@hotmail.com,423456789.0,QLD,4101.0,James,O'Connor,...,423456789.0,NSW,2010.0,False,False,False,False,False,True,True
4,1005,Jessica,Brown,1999-02-28,jess.brown@yahoo.com,456789012.0,QLD,4350.0,Jessica,Brown-Taylor,...,456789012.0,QLD,4350.0,False,True,False,True,False,False,False
6,1007,Olivia,Smith,2000-08-30,olivia.smith@outlook.com,NaN,QLD,4000.0,Olivia,Smith,...,478901234.0,QLD,4000.0,False,False,False,False,True,False,False
9,1010,Noah,Garcia,2001-01-25,noah.garcia@hotmail.com,401234567.0,QLD,4500.0,Noah,Garcia,...,401234567.0,QLD,4500.0,False,False,False,False,False,False,False


In [45]:
# SUmmary of changed applicants 

existing_applicants["has_any_change"] = existing_applicants[change_columns].any(axis=1)

changed_applicants = existing_applicants[
    existing_applicants["has_any_change"]
]

unchanged_applicants = existing_applicants[
    ~existing_applicants["has_any_change"]
]

new_applicants = applicant_comparison[
    applicant_comparison["_merge"] == "right_only"
]

print(f"Changed existing applicants: {len(changed_applicants)}")
print(f"Unchanged existing applicants: {len(unchanged_applicants)}")
print(f"New applicants in update file: {len(new_applicants)}")

print("\nChanged applicants:")
print(changed_applicants["applicant_id"].tolist())

print("\nUnchanged applicants:")
print(unchanged_applicants["applicant_id"].tolist())

print("\nNew applicants:")
print(new_applicants["applicant_id"].tolist())

Changed existing applicants: 3
Unchanged existing applicants: 1
New applicants in update file: 1

Changed applicants:
[1002, 1005, 1007]

Unchanged applicants:
[1010]

New applicants:
[1016]


In [46]:
# Accepted offer check for gold layer logic

accepted_preferences = df_preferences[
    df_preferences["response"].str.lower() == "accepted"
].copy()

accepted_preferences.sort_values(
    by=["applicant_id", "preference_order"]
)

,preference_id,applicant_id,course_code,preference_order,application_year,offer_status,offer_date,response,response_date
0,P001,1001,UQ-COMP1,1,2025,Offered,2025-03-15,Accepted,2025-03-18
3,P004,1002,QUT-BS001,2,2025,Offered,2025-03-22,Accepted,2025-03-25
21,P022,1002,QUT-BS001,2,2025,Offered,2025-03-22,Accepted,2025-03-25
5,P006,1003,UQ-ENG01,2,2025,Offered,2025-03-15,Accepted,2025-03-16
6,P007,1004,UQ-ENG01,1,2025,Offered,2025-03-15,Accepted,2025-03-17
7,P008,1005,GU-NURS1,1,2025,Offered,2025-03-15,Accepted,2025-03-19
8,P009,1006,USQ-IT001,1,2025,Offered,2025-03-15,Accepted,2025-03-16
12,P013,1009,GU-PSYC1,1,2025,Offered,2025-03-15,Accepted,2025-03-16
14,P015,1011,GU-EDU01,1,2025,Offered,2025-03-15,Accepted,2025-03-18
15,P016,1012,UQ-ARTS1,1,2025,Offered,2025-03-15,Accepted,2025-03-20


In [47]:
# Highest accepted offer per applicant

highest_accepted_preferences = (
    accepted_preferences
    .sort_values(by=["applicant_id", "preference_order"])
    .drop_duplicates(subset=["applicant_id"], keep="first")
)

highest_accepted_preferences

,preference_id,applicant_id,course_code,preference_order,application_year,offer_status,offer_date,response,response_date
0,P001,1001,UQ-COMP1,1,2025,Offered,2025-03-15,Accepted,2025-03-18
3,P004,1002,QUT-BS001,2,2025,Offered,2025-03-22,Accepted,2025-03-25
5,P006,1003,UQ-ENG01,2,2025,Offered,2025-03-15,Accepted,2025-03-16
6,P007,1004,UQ-ENG01,1,2025,Offered,2025-03-15,Accepted,2025-03-17
7,P008,1005,GU-NURS1,1,2025,Offered,2025-03-15,Accepted,2025-03-19
8,P009,1006,USQ-IT001,1,2025,Offered,2025-03-15,Accepted,2025-03-16
12,P013,1009,GU-PSYC1,1,2025,Offered,2025-03-15,Accepted,2025-03-16
14,P015,1011,GU-EDU01,1,2025,Offered,2025-03-15,Accepted,2025-03-18
15,P016,1012,UQ-ARTS1,1,2025,Offered,2025-03-15,Accepted,2025-03-20
16,P017,1013,UQ-COMP1,1,2025,Offered,2025-03-15,Accepted,2025-03-16


In [ ]:
for name, df in df_dict.items():
    missing_counts = df.isna().sum()
    missing_counts = missing_counts[missing_counts > 0]

    print(f"\nMissing values in {name}:")
    if missing_counts.empty:
        print("No missing values")
    else:
        print(missing_counts)